# Meqpy Tuturial

In [ ]:
import meqpy

import numpy as np
import matplotlib.pyplot as plt

- [2. System and States](#system_and_states)
    - [2.1 Helper Functions](#system_and_states_helper)
    - [2.2 Charging Rates](#system_and_states_charging_rates)
        - [2.2.1 Normalized Charging Transitions](#system_and_states_normalized_charging_transitions)
            - [Lineshapes and HWHM](#system_and_states_lineshape_hwhm)
            - [Shift by reorganisation energy](#system_and_states_reorg_shift)
        - [2.2.2 Coupling Strength](#system_and_states_coupling_strength)
            - [kappa_mode and workfunction](#system_and_states_kappa_workfunction)
    - [2.3 Spin Selection Rule](#system_and_states_spin_selection_rule)


<a id='system_and_states'></a>
## 2) System and States

The rate-equation (or master-equation) model used in ``meqpy`` is built on a Markov chain, in which a ``System`` consists of various ``States`` that the system can occupy. In addition, there are transition probabilities between those states.

Here, the various states correspond to electron configurations, e.g. the ground state, an ionic state, or an excited state. Transitions between states can occur via charging events (electron attachment or removal) or via charge-neutral transitions, e.g. radiative decay or excitation. ``meqpy`` is designed to handle the charging transitions in particular.

To create a state, use the ``meqpy.State`` class, which takes the following parameters:
- label : Label for the state (must be unique within the system)
- energy : Energy of the state in eV. The sign is independent of the charge state, so the ground state should have the lowest energy.
- charge : Charge of the state.
- multiplicity : Multiplicity of the state; the default is 1.

In the following, three states are created: the neutral ground state, as well as the anionic and cationic charge states. Note that both ionic states have positive energy, even though they have different charge states.

In [ ]:
neutral = meqpy.State(label="neutral", energy=0.0, charge=0, multiplicity=1)
anion = meqpy.State(label="anion", energy=1.0, charge=-1, multiplicity=2)
cation = meqpy.State(label="cation", energy=0.5, charge=+1, multiplicity=2)

The ``System`` class in ``meqpy`` consists of various states and is responsible for constructing the master equation that needs to be solved. This allows for fast and straightforward design of STM experiments in silico.

There are three ways to add states to a System object:

In [ ]:
# Option 1: add states one by one
system = meqpy.System()
system.add_state(neutral)
system.add_state(anion)
system.add_state(cation)

# Option 2: add states during initiation
system = meqpy.System(states=[neutral, anion, cation])

# Option 3: write list of states directly into System
system = meqpy.System()
system.states = [neutral, anion, cation]

<a id='system_and_states_helper'></a>
### 2.1) System and States: Helper Functions

The ``System`` class comes with a variety of helper function:

Properties:
- ``num_states``: number of states in the system
- ``shape``: shape of the rate-equation matrix: (num_states, num_states)
- ``energies``: list of energies of all states
- ``charges``: list of charges of all states
- ``multiplicities``: list of multiplicities of all states
- ``dE``: (num_states, num_states) matrix, with each entry dE<sub>fi</sub> = E<sub>f</sub> - E<sub>i</sub> being the energy difference between initial and final state
- ``dQ``: (num_states, num_states) matrix, with each entry dQ<sub>fi</sub> = Q<sub>f</sub> - Q<sub>i</sub> being the charge difference between initial and final state
- ``dM``: (num_states, num_states) matrix, with each entry dM<sub>fi</sub> = M<sub>f</sub> - M<sub>i</sub> being the multiplicity difference between initial and final state
- ``clebsch_gordan_factors``: (num_states, num_states) matrix, with each entry corresponding to the renorming factor of transition probability, caused by the multiplicities
- ``zeros``: (num_states, num_states) matrix with all entries being zero
- ``ones``: (num_states, num_states) matrix with all entries being one

In addition there are additional methods, described below:
- ``get_state()``
- ``get_index()``
- ``matrix_by_states()``
- ``rescale_by_states()``

In [ ]:
# returns state by label or index
state_by_index = system.get_state(0)
state_by_label = system.get_state("neutral")

state_by_index == state_by_label

In [ ]:
# get index of State in System by its label
system.get_index("anion")

In [ ]:
# create matrix with all entries zero, except transitions from initial to final state
# the function accepts labels as well as indices
system.matrix_by_states("anion", "neutral")

In [ ]:
# matrix_by_states can also return a symmetric matrix
system.matrix_by_states("anion", "neutral", symmetric=True)

In [ ]:
# create matrix with all entries one, except transitions from initial to final state
# instead the entry is rescaling factor
system.rescale_by_states("neutral", "cation", 2.0)

In [ ]:
# this can also be made symmetric
system.rescale_by_states("neutral", "cation", 0.5, symmetric=True)

<a id='system_and_states_charging_rates'></a>
### 2.2) System and States: Charging Rates

The most important method of the ``System`` class is ``System.charging_rates()``. It automatically calculates the transition rates between all states due to charging via tunneling. These rates depend on the bias voltage (``bias``) applied between the system and the lead, while the coupling strength is given by the tunneling distance ``z`` in Angstrom.

In [ ]:
z = 5.0  # Angstrom
bias = 1.0  # Volt
system.charging_rates(z, bias)

``System.charging_rates()`` accepts both ``float`` and ``np.ndarray`` as inputs for ``z`` and ``bias`` and returns an array of shape ``(K, N, M, M)``, with ``K`` being number of barrier widths (or tip height) ``z``, ``N`` being number of bias values and ``M`` is the number of states in the system. By default, any dimension with length 1 will be squeezed, but setting ``squeeze = False`` will prohibit this.

In [ ]:
shape = system.charging_rates(5.0, 0.0).shape
print(f"K = 1, N = 1: shape = {shape}")

shape = system.charging_rates(5.0, np.arange(10)).shape
print(f"K = 1, N = 10: shape = {shape}")

shape = system.charging_rates(np.arange(5), 0.0, squeeze=False).shape
print(f"K = 5, N = 1, squeeze = False: shape = {shape}")

shape = system.charging_rates(np.arange(5), np.arange(10)).shape
print(f"K = 5, N = 10: shape = {shape}")

``System.charging_rates()`` is a wrapper function to return the product of:
- ``System.normalized_charge_transition()``
- ``System.coupling_strength()``
- ``System.clebsch_gordan_factors``

<a id='system_and_states_normalized_charging_transitions'></a>
#### 2.2.1) Normalized Charging Transitions

The ``System.normalized_charging_transitions()`` method captures the energy dependence of the tunneling probability, due to Fermi-Dirac distribution. It returns for each combination of initial and final state a (broadened) a bias voltage dependent step-function, shifted and flipped by the energy and charge difference between the two states.



In [ ]:
bias = 0.0
norm_transition = system.normalized_charging_transitions(bias)
print(f"Bias = {bias}:\n{norm_transition}\n")

bias = np.linspace(-2, 2, 201)
norm_transition = system.normalized_charging_transitions(bias)
print(f"Shape of output for {len(bias)} bias values:\n{norm_transition.shape}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3))

for i in range(2):
    a = system.get_state(0).label
    b = system.get_state(i + 1).label
    ax[i].plot(bias, norm_transition[:, i + 1, 0], label=f"{a} → {b}")
    ax[i].plot(bias, norm_transition[:, 0, i + 1], label=f"{b} → {a}")
    ax[i].legend(frameon=False)
    ax[i].set_title(f"{b} ↔ {a}")
    ax[i].set_xlabel("Bias Voltage (V)")
    ax[i].set_ylabel("Norm. Charge Transition")
plt.tight_layout(pad=1)
plt.show()

<a id='system_and_states_lineshape_hwhm'></a>
##### 2.2.1a) Lineshape and HWHM

As step-function, one can choose between three different lineshapes, corresponding to the derivative of the step function:
- "gaussian" (default)
- "lorentzian"
- "dirac"
The lineshape can be set during initialization of the system
```python
system = meqpy.System(lineshape = <str>)
```
or afterwards
```python
system.lineshape = <str>
```

In addition, the broadening for the "gaussian" and "lorentzian" lineshape can be set via the ``hwhm`` parameter. If ``hwhm == 0``, the "dirac" lineshape will be used.


In [ ]:
bias = np.linspace(-2, 2, 201)
system.hwhm = 50e-3  # eV

system.lineshape = "gaussian"
gaussian_transition = system.normalized_charging_transitions(bias)

system.lineshape = "lorentzian"
lorentzian_transition = system.normalized_charging_transitions(bias)

system.lineshape = "dirac"
dirac_transition = system.normalized_charging_transitions(bias)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 3))

i, j = 0, 1
a = system.get_state(i).label
b = system.get_state(j).label

ax[0].plot(bias, gaussian_transition[:, i, j], label=f"{a} → {b}")
ax[0].plot(bias, gaussian_transition[:, j, i], label=f"{b} → {a}")
ax[0].set_title("gaussian")

ax[1].plot(bias, lorentzian_transition[:, i, j], label=f"{a} → {b}")
ax[1].plot(bias, lorentzian_transition[:, j, i], label=f"{b} → {a}")
ax[1].set_title("lorentzian")

ax[2].plot(bias, dirac_transition[:, i, j], label=f"{a} → {b}")
ax[2].plot(bias, dirac_transition[:, j, i], label=f"{b} → {a}")
ax[2].set_title("dirac")

for i in range(3):
    ax[i].set_xlabel("Bias Voltage (V)")
    ax[i].set_ylabel("Norm. Charge Transition")
    ax[i].legend(frameon=False)
plt.tight_layout(pad=1)
plt.show()

<a id='system_and_states_reorg_shift'></a>
##### 2.2.1b) Shift by Reorganisation Energy

The effective shift of ion resonances due to a structural reorganisation energy, i.e. for molecules on salt decoupling layers, can be roughly mimicked by using the ``reorg_shift`` parameter. The step-function will be shifted accordingly to higher energies (NOT higher bias voltages).

In [ ]:
bias = np.linspace(-2, 2, 201)
system.hwhm = 100e-3  # eV
system.lineshape = "gaussian"

system.reorg_shift = 0.0  # eV
transition_wo_shift = system.normalized_charging_transitions(bias)

system.reorg_shift = 0.25  # eV
transition_w_shift = system.normalized_charging_transitions(bias)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))

i, j = 0, 1
a = system.get_state(i).label
b = system.get_state(j).label

ax[0].plot(bias, transition_wo_shift[:, i, j], label=f"{a} → {b}")
ax[0].plot(bias, transition_wo_shift[:, j, i], label=f"{b} → {a}")
ax[0].set_title("reorg_shift: 0 eV")

ax[1].plot(bias, transition_w_shift[:, i, j], label=f"{a} → {b}")
ax[1].plot(bias, transition_w_shift[:, j, i], label=f"{b} → {a}")
ax[1].set_title("reorg_shift: 0.25 eV")

for i in range(2):
    ax[i].set_xlabel("Bias Voltage (V)")
    ax[i].set_ylabel("Norm. Charge Transition")
    ax[i].legend(frameon=False)
plt.tight_layout(pad=1)
plt.show()

<a id='system_and_states_coupling_strength'></a>
#### 2.2.2) Coupling Strength

The coupling strength captures how strong the system is coupled to the electron bath of the lead (i.e. sample or tip), in dependence of barrier width (and height):
```math
\mathrm{coupling\_strength} = G_0 \cdot e^{ -2 \kappa  z },
```
with ``kappa`` ($\kappa$) being the decay constant (more below) and ``z`` the barrier width. The prefactor ``G0 = e/h`` represents the tunneling rate corresponding to the single-spin-channel conductance quantum, with ``e`` being the elementary charge and ``h`` the Planck constant.


In [ ]:
z = np.arange(3.0, 9.0, 0.5)
coupling_strength = system.coupling_strength(z)
coupling_strength.shape

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))

ax.plot(z, coupling_strength[:, 1, 0])
ax.set_yscale("log")
ax.set_xlabel("z (Å)")
ax.set_ylabel("Coupling Strength (1/s)")

plt.show()

<a id='system_and_states_kappa_workfunction'></a>
##### 2.2.2a) kappa_mode and workfunction

The ``System`` class has three different levels of approximation to calculate the decay constant ``kappa``, which can be selected via the ``kappa_mode`` parameter:
- ``10``: kappa is set, such that a increase (decrease) of the tunneling barrier width of 1Å will decrease (increase) the coupling strength by a factor of 10.
- ``constant``: the tunneling barrier height corresponds to the workfunction of the system
- ``full``: the tunneling barrier height is given by the workfunction of the system plus the energy difference between final and initial state. In addition the effect of the applied bias voltage on the barrier heigth will be considered as well. (default)

The ``kappa_mode`` parameter can be either set for the whole System:
```python
system = meqpy.System(kappa_mode = <str>)  # option 1
syste.kappa_mode = <str>  # option 2
```
but it can also be used locally when calling ``System.coupling_strength()`` or  ``System.charging_rates()``.

The worfunction of the system can be set via the ``workfunction`` parameter:
```python
system = meqpy.System(workfunction = <float>)  # option 1
system.workfunction = <float>  # option 2
```

In [ ]:
z = 5.0  # Å
bias = np.linspace(-2, 2, 201)  # V

G0 = meqpy.constants.G0

# kappa_mode: 10
coupling_10 = system.coupling_strength(z, bias, kappa_mode="10") / G0

# kappa_mode: constant
coupling_constant = system.coupling_strength(z, bias, kappa_mode="constant") / G0

# kappa_mode: full with 4eV workfunction
system.workfunction = 4.0  # eV
coupling_full_4 = system.coupling_strength(z, bias, kappa_mode="full") / G0

# kappa_mode: full with 5eV workfunction
system.workfunction = 5.0  # eV
coupling_full_5 = system.coupling_strength(z, bias, kappa_mode="full") / G0

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 3))

i, j = 0, 2
a = system.get_state(i).label
b = system.get_state(j).label

ax[0].plot(bias, coupling_10[:, j, i] * 1e5, label=f"{a} → {b}")
ax[0].plot(bias, coupling_10[:, i, j] * 1e5, label=f"{b} → {a}")
ax[0].set_title('"10"')

ax[1].plot(bias, coupling_constant[:, j, i] * 1e5, label=f"{a} → {b}")
ax[1].plot(bias, coupling_constant[:, i, j] * 1e5, label=f"{b} → {a}")
ax[1].set_title('"constant"')

ax[2].plot(bias, coupling_full_4[:, j, i] * 1e5, label=f"{a} → {b} @ 4eV")
ax[2].plot(bias, coupling_full_4[:, i, j] * 1e5, label=f"{b} → {a} @ 4eV")
ax[2].plot(
    bias, coupling_full_5[:, j, i] * 1e5, "--", label=f"{a} → {b} @ 5eV", color="C0"
)
ax[2].plot(
    bias, coupling_full_5[:, i, j] * 1e5, "--", label=f"{b} → {a} @ 5eV", color="C1"
)
ax[2].set_title('"full"')

for i in range(3):
    ax[i].set_xlabel("Bias Voltage (V)")
    ax[i].set_ylabel(r"Coupling Strength (G0 $\cdot 10^{-5}$)")
    ax[i].legend(frameon=False)
plt.tight_layout(pad=1)
plt.show()

<a id='system_and_states_spin_selection_rule'></a>
### 2.3) Spin Selection Rule

By default, the ``System`` class imposes a spin selection rule that forbids any charging transition between states whose multiplicity differs by anything other than one. However, in some systems with high spin-orbit coupling, spin is no longer a good quantum number, and thus the spin selection rule no longer applies. For this reason, the spin selection rule can be switched off via the ``spin_selection_rule`` parameter.

In [ ]:
# create singlet and quartet state
singlet = meqpy.State(label="neutral", energy=0.0, charge=0, multiplicity=1)
quartet = meqpy.State(label="quartet", energy=1.0, charge=-1, multiplicity=4)

soc_system = meqpy.System(states=[singlet, quartet], spin_selection_rule=True)

# all transitions between singlet and quartet are spin forbidden:
soc_system.charging_rates(z=5.0, bias=0.0)

In [ ]:
# disable spin selection rule -> transitions are allowed
soc_system.spin_selection_rule = False
soc_system.charging_rates(z=5.0, bias=0.0)